# Project 4 (FinQA rerun): Query Transformation Engine

Reruns Project 4's five techniques on your **real** FinQA data — 883
questions, 8,931 row-level chunks — with **real** `qwen2.5:7b-instruct` via
Ollama and **real** `BAAI/bge-small-en-v1.5` embeddings, using your existing
`src/` modules (`src.models`, `src.utils.io`, `src.eval.retrieval_harness`,
`src.eval.sampling`) wherever they already do the job.

**Prerequisites**
```bash
ollama serve                       # in a separate terminal
ollama pull qwen2.5:7b-instruct
pip install langchain-classic      # MultiQueryRetriever, HypotheticalDocumentEmbedder,
                                    # create_history_aware_retriever now live here
```
Run this notebook from the project root (same level as `src/`, `data/`, `scripts/`).

**New files this notebook uses** (added under `src/query_transform/`, same
layout as your existing `src/chunking/`, `src/eval/`, `src/rag/`):
`fusion.py`, `hyde.py`, `multi_query.py`, `step_back.py`, `decompose.py`, `rewrite.py`


In [16]:
import sys, os, time, json
from pathlib import Path

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("OMP_NUM_THREADS", "1")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent   # if running from notebooks/
sys.path.insert(0, str(PROJECT_ROOT))

from src.models import load_embedder, load_llm
from src.utils.io import load_jsonl, save_json
from src.eval.retrieval_harness import (
    load_index_chunks, context_recall_at_k, context_precision_at_k, word_overlap_relevance,
)
from src.eval.sampling import stratified_sample_by_op, describe_sample

from src.query_transform.fusion import reciprocal_rank_fusion
from src.query_transform.hyde import build_hyde_embedder, hyde_search
from src.query_transform.multi_query import build_multi_query_retriever, multi_query_search
from src.query_transform.step_back import build_step_back_chain, step_back_search
from src.query_transform.decompose import build_decompose_chain, decompose_search
from src.query_transform.rewrite import build_history_aware_retriever, rewrite_search

print("imports OK")


imports OK


## 0. Concept — building the real index

Same row-level `chunks.jsonl` your `eval_dataset.jsonl`'s `gold_chunk_ids`
were annotated against (the scheme `evaluate_chunking_strategy` in your own
`retrieval_harness.py` already uses for exact-id scoring) — **not**
`chunks_whole_table.jsonl`, which is for generation, not this retrieval
benchmark.

Using `QdrantVectorStore` with an in-memory client — same library class your
`SimpleRAG` pipeline uses, just without touching disk, so this notebook can
be rerun freely without cleaning up a collection each time.


In [17]:
from langchain_core.documents import Document
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

chunks = load_index_chunks("data/processed/chunks.jsonl")   # is_noise filtered, matches gold id scheme
eval_examples = load_jsonl("data/processed/eval_dataset.jsonl")
print(f"{len(chunks)} indexable chunks, {len(eval_examples)} eval questions")

embeddings = load_embedder()
llm = load_llm()   # qwen2.5:7b-instruct, temperature=0.0 (src/models.py default)

docs = [Document(page_content=c["text"], metadata={"chunk_id": c["chunk_id"], "doc_id": c["doc_id"]})
        for c in chunks]

client = QdrantClient(":memory:")
dim = len(embeddings.embed_query("dimension probe"))
client.create_collection(collection_name="finqa_query_transform",
                          vectors_config=VectorParams(size=dim, distance=Distance.COSINE))
vectorstore = QdrantVectorStore(client=client, collection_name="finqa_query_transform", embedding=embeddings)

t0 = time.time()
vectorstore.add_documents(docs, batch_size=256)
print(f"indexed {len(docs)} chunks in {time.time()-t0:.1f}s")


8532 indexable chunks, 883 eval questions


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

indexed 8532 chunks in 92.2s


## 1. Building the eval sample + failure-mode tags

883 questions × 5 techniques × 1+ LLM call each is a lot of local Ollama
calls. Reusing your **own** `stratified_sample_by_op` (`src/eval/sampling.py`)
— built for exactly this problem (your generation eval already uses it for
the same reason) — keeps the sample small but representative across all 9
FinQA operation types (subtract/divide dominate the full set; a uniform
sample would barely touch `table_min`/`table_max`).

Failure-mode tags, derived from your real data rather than hand-labeled like
the Nimbus set:
- **multihop**: `len(gold_chunk_ids) >= 2` — a real, common case (396/818
  scored questions, ~48%) — FinQA's own `program` field chaining two
  different reported figures together.
- **vocab**: <15% word overlap between the question and its own gold chunk
  text — genuine phrasing mismatch (e.g. casual question wording vs.
  `"Payments Volume (billions) is $2,457"` table-row phrasing).
- **control**: single gold chunk id + decent overlap — the easy case.
- **vague**: FinQA questions are template-generated and always ask for a
  specific computed figure — there's no natural "vague" query in this
  dataset the way there was in the synthetic Nimbus corpus. Skipped here;
  this is an honest gap, not an oversight.


In [20]:
import re

def word_overlap_ratio(a: str, b: str) -> float:
    a_words = {w.lower().strip('.,;:()%$') for w in a.split()}
    b_words = {w.lower().strip('.,;:()%$') for w in b.split()}
    if not a_words:
        return 0.0
    return len(a_words & b_words) / len(a_words)

def tag_failure_mode(ex: dict, gold_text_lookup: dict) -> str:
    gold_ids = ex["gold_chunk_ids"]
    if len(gold_ids) >= 2:
        return "multihop"
    gold_texts = [gold_text_lookup[g] for g in gold_ids if g in gold_text_lookup]
    if gold_texts and max(word_overlap_ratio(ex["question"], gt) for gt in gold_texts) < 0.15:
        return "vocab"
    return "control"

gold_text_lookup = {c["chunk_id"]: c["text"] for c in chunks}

sample = stratified_sample_by_op(eval_examples, per_op_n=10, seed=42)
sample = [ex for ex in sample if ex["gold_chunk_ids"]]   # drop the 65 unscoreable questions
for ex in sample:
    ex["failure_mode"] = tag_failure_mode(ex, gold_text_lookup)

print(f"sample size: {len(sample)}")
for op, n, avail in describe_sample(eval_examples, per_op_n=10):
    print(f"  {op:15s} {n}/{avail}")

from collections import Counter
print("\nfailure-mode composition:", Counter(ex["failure_mode"] for ex in sample))


sample size: 64
  divide          10/323
  subtract        10/360
  add             10/100
  multiply        10/56
  table_average   10/19
  table_sum       4/4
  greater         9/9
  table_min       5/5
  table_max       7/7

failure-mode composition: Counter({'control': 34, 'multihop': 24, 'vocab': 6})


## 2. Baseline (no transformation)

Reusing `context_recall_at_k` / `context_precision_at_k` from your own
`retrieval_harness.py` rather than rewriting metric code — same functions
your Project 3 vector-DB shootout already validated.


In [19]:
def evaluate_run(run_fn, sample, k=5):
    from collections import defaultdict
    per_mode = defaultdict(list)
    rows = []
    for ex in sample:
        t0 = time.time()
        retrieved = run_fn(ex["question"])
        latency = time.time() - t0
        gold_ids = set(ex["gold_chunk_ids"])
        recall = context_recall_at_k(retrieved, gold_ids, k)
        precision = context_precision_at_k(retrieved, gold_ids, k)
        per_mode[ex["failure_mode"]].append((recall, precision, latency))
        rows.append({"id": ex["id"], "question": ex["question"], "mode": ex["failure_mode"],
                      "recall": recall, "precision": precision, "latency_s": latency})
    summary = {}
    for mode, vals in per_mode.items():
        n = len(vals)
        summary[mode] = {"n": n,
                          "recall": sum(v[0] for v in vals) / n,
                          "precision": sum(v[1] for v in vals) / n,
                          "avg_latency_s": sum(v[2] for v in vals) / n}
    all_vals = [v for vals in per_mode.values() for v in vals]
    n = len(all_vals)
    summary["ALL"] = {"n": n,
                       "recall": sum(v[0] for v in all_vals) / n,
                       "precision": sum(v[1] for v in all_vals) / n,
                       "avg_latency_s": sum(v[2] for v in all_vals) / n}
    return summary, rows


def baseline_run(question, k=5):
    docs = vectorstore.similarity_search(question, k=k)
    return [d.metadata["chunk_id"] for d in docs]

baseline_summary, baseline_rows = evaluate_run(baseline_run, sample)
print(json.dumps(baseline_summary, indent=2))


{
  "multihop": {
    "n": 24,
    "recall": 0.375,
    "precision": 0.19166666666666668,
    "avg_latency_s": 0.02278536558151245
  },
  "control": {
    "n": 34,
    "recall": 0.6470588235294118,
    "precision": 0.12941176470588237,
    "avg_latency_s": 0.02146007032955394
  },
  "vocab": {
    "n": 6,
    "recall": 0.16666666666666666,
    "precision": 0.03333333333333333,
    "avg_latency_s": 0.022030353546142578
  },
  "ALL": {
    "n": 64,
    "recall": 0.5,
    "precision": 0.14375000000000002,
    "avg_latency_s": 0.022010520100593567
  }
}


## 3. HyDE

**Concept.** Embed the LLM's hypothetical *answer* passage instead of the
raw question — answer-shaped text sits closer to answer-shaped table/passage
text than a question does. Using LangChain's `HypotheticalDocumentEmbedder`
with its built-in **`fiqa`** prompt (`"Please write a financial article
passage to answer the question..."`) — a real library prompt template tuned
for financial QA, not hand-written for this notebook.

**Architecture.** `question → HyDE LLM chain → hypothetical passage →
embed → similarity_search_by_vector → top-k`

**Where this should win**: the `vocab` category — casual question phrasing
vs. formal table-row/passage language is exactly HyDE's target failure mode.


In [ ]:
hyde_embedder = build_hyde_embedder(llm, embeddings)
def hyde_run(question, k=5):
    return hyde_search(vectorstore, question, hyde_embedder, k=k)

hyde_summary, hyde_rows = evaluate_run(hyde_run, sample)
print(json.dumps(hyde_summary, indent=2))


{
  "multihop": {
    "n": 24,
    "recall": 0.125,
    "precision": 0.075,
    "avg_latency_s": 21.025333672761917
  },
  "control": {
    "n": 34,
    "recall": 0.4411764705882353,
    "precision": 0.08823529411764706,
    "avg_latency_s": 20.644519763834335
  },
  "vocab": {
    "n": 6,
    "recall": 0.0,
    "precision": 0.0,
    "avg_latency_s": 21.248150666554768
  },
  "ALL": {
    "n": 64,
    "recall": 0.28125,
    "precision": 0.07500000000000001,
    "avg_latency_s": 20.84391537681222
  }
}


## 4. Multi-Query Retrieval

**Concept.** Generate several reworded versions of the question, retrieve
for each, take the union of unique chunks. Using LangChain's
`MultiQueryRetriever` directly — no hand-rolled paraphrase generation or
fusion; this is the library's real default behavior (union + dedupe, not
RRF-fused, unlike the hand-built Nimbus version).

**Architecture.** `question → MultiQueryRetriever (LLM generates N variants
internally) → union of retrieved docs across variants → dedupe`

**Where this should win**: `multihop` — FinQA's compound questions (two
different reported figures combined) benefit from several angles of attack,
same pattern as the Nimbus result.


In [ ]:
mq_retriever = build_multi_query_retriever(vectorstore, llm, k=5)

def multi_query_run(question, k=5):
    return multi_query_search(mq_retriever, question)[:k]

mq_summary, mq_rows = evaluate_run(multi_query_run, sample)
print(json.dumps(mq_summary, indent=2))


{
  "multihop": {
    "n": 24,
    "recall": 0.3333333333333333,
    "precision": 0.16666666666666666,
    "avg_latency_s": 7.200440774361293
  },
  "control": {
    "n": 34,
    "recall": 0.6470588235294118,
    "precision": 0.12941176470588237,
    "avg_latency_s": 7.172841899535236
  },
  "vocab": {
    "n": 6,
    "recall": 0.16666666666666666,
    "precision": 0.03333333333333333,
    "avg_latency_s": 7.505915284156799
  },
  "ALL": {
    "n": 64,
    "recall": 0.484375,
    "precision": 0.134375,
    "avg_latency_s": 7.214417107403278
  }
}


## 5. Step-Back Prompting

**Concept.** Generalize a narrow question one level up before retrieving,
fuse with the original via RRF. No official LangChain class for this one —
built as a standard LCEL chain (`ChatPromptTemplate | llm | StrOutputParser`),
with in-domain financial few-shot examples (generic examples produce
step-back questions too generic to be useful against report tables).

**Architecture.** `question → LLM generalizes → RRF-fuse(search(question),
search(general_question)) → top-k`

**Recall the Nimbus lesson**: this technique actively hurt already-broad
queries. FinQA's questions are all narrow/computational by construction
(never vague), so this is actually a more favorable setting for step-back
than Nimbus was — worth checking whether that holds.


In [ ]:
step_back_chain = build_step_back_chain(llm)

def step_back_run(question, k=5):
    return step_back_search(vectorstore, question, step_back_chain, k=k)

step_back_summary, step_back_rows = evaluate_run(step_back_run, sample)
print(json.dumps(step_back_summary, indent=2))


{
  "multihop": {
    "n": 24,
    "recall": 0.34722222222222227,
    "precision": 0.17500000000000002,
    "avg_latency_s": 2.3705022732416787
  },
  "control": {
    "n": 34,
    "recall": 0.6470588235294118,
    "precision": 0.12941176470588237,
    "avg_latency_s": 2.2341961229548737
  },
  "vocab": {
    "n": 6,
    "recall": 0.16666666666666666,
    "precision": 0.03333333333333333,
    "avg_latency_s": 2.3796446720759072
  },
  "ALL": {
    "n": 64,
    "recall": 0.4895833333333333,
    "precision": 0.1375,
    "avg_latency_s": 2.2989467307925224
  }
}


## 6. Sub-Query Decomposition

**Concept.** Split a compound question into independent sub-questions,
retrieve each separately, RRF-fuse. Uses `llm.with_structured_output`
(Pydantic `SubQuestions` schema, real function-calling/JSON-mode under the
hood) — the LLM can't return anything except a valid schema instance, no
free-text list parsing needed.

**Architecture.** `question → LLM.decompose() → [sub_q1, sub_q2, ...] →
RRF-fuse(search(sub_q1), search(sub_q2), ...) → top-k`

**Where this should win**: `multihop`, with (per the Nimbus result) close to
zero side effects elsewhere, since the schema explicitly allows returning
the original question unchanged when it isn't actually compound.


In [ ]:
decompose_chain = build_decompose_chain(llm)

def decompose_run(question, k=5):
    return decompose_search(vectorstore, question, decompose_chain, k=k)

decompose_summary, decompose_rows = evaluate_run(decompose_run, sample)
print(json.dumps(decompose_summary, indent=2))


{
  "multihop": {
    "n": 24,
    "recall": 0.4375,
    "precision": 0.21666666666666667,
    "avg_latency_s": 5.442034433285396
  },
  "control": {
    "n": 34,
    "recall": 0.6176470588235294,
    "precision": 0.12352941176470589,
    "avg_latency_s": 5.007977913407719
  },
  "vocab": {
    "n": 6,
    "recall": 0.16666666666666666,
    "precision": 0.03333333333333333,
    "avg_latency_s": 5.41244622071584
  },
  "ALL": {
    "n": 64,
    "recall": 0.5078125,
    "precision": 0.15,
    "avg_latency_s": 5.208668012171984
  }
}


## 7. Query Rewriting (bonus — conversational)

**Honest framing**: FinQA's 883 questions are standalone, no multi-turn
structure, so there's no way to measure this failure mode *within* the
benchmark. But the moment this index sits behind a chat UI instead of a
single-shot eval script, users will ask follow-ups. This section validates
the mechanism against real financial question phrasing using a small,
hand-built conversational sample **grounded in real questions/answers from
your own `eval_dataset.jsonl`** (not invented from scratch).

Uses LangChain's `create_history_aware_retriever` — built for exactly this;
it skips the LLM call entirely when there's no history or the input is
already standalone.


In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

# Grounded in real eval_dataset.jsonl entries -- turn 1 is a real question,
# the follow-up and its gold chunk id are hand-authored to test whether raw
# vs. rewritten retrieval finds the right row.
CONVERSATIONAL_SAMPLE = [
    {
        "history": [HumanMessage("What is the average payment volume per transaction for American Express?"),
                    AIMessage("It was 127.40 in 2008.")],
        "raw_followup": "How does that compare to MasterCard?",
        "gold_chunk_ids": ["V/2008/page_17.pdf::table_row::2"],
    },
    {
        "history": [HumanMessage("What was the change in millions of operating income from 2016 to 2017?"),
                    AIMessage("It increased by $688 million.")],
        "raw_followup": "What was the actual 2017 figure?",
        "gold_chunk_ids": ["PM/2017/page_38.pdf::table_row::3"],
    },
    {
        "history": [HumanMessage("What was the increase in Class A common stock issued and outstanding between years, in thousands?"),
                    AIMessage("It increased by 995 thousand shares.")],
        "raw_followup": "What was the value in the earlier year?",
        "gold_chunk_ids": ["CME/2017/page_97.pdf::table_row::2"],
    },
    {
        "history": [HumanMessage("What was the change in weighted average common shares outstanding for diluted computations from 2012 to 2013, in millions?"),
                    AIMessage("It decreased by 1.9 million.")],
        "raw_followup": "Did it go up or down?",
        "gold_chunk_ids": ["LMT/2013/page_74.pdf::table_row::3"],
    },
    {
        "history": [HumanMessage("In millions, what was the total residential mortgages balance for 2013 and 2012?"),
                    AIMessage("It was $3,576 million combined.")],
        "raw_followup": "What about just 2012?",
        "gold_chunk_ids": ["PNC/2013/page_62.pdf::table_row::6"],
    },
]


history_aware_retriever = build_history_aware_retriever(vectorstore, llm, k=5)

raw_hits, rewritten_hits = 0, 0
for conv in CONVERSATIONAL_SAMPLE:
    gold = set(conv["gold_chunk_ids"])

    raw_retrieved = [d.metadata["chunk_id"] for d in vectorstore.similarity_search(conv["raw_followup"], k=5)]
    raw_ok = bool(gold & set(raw_retrieved))
    raw_hits += raw_ok

    rewritten_retrieved = rewrite_search(history_aware_retriever, conv["raw_followup"], conv["history"])
    rewritten_ok = bool(gold & set(rewritten_retrieved))
    rewritten_hits += rewritten_ok

    print(f"{'OK ' if raw_ok else 'ERR'} raw       | \"{conv['raw_followup']}\"")
    print(f"{'OK ' if rewritten_ok else 'ERR'} rewritten | (resolved via history_aware_retriever)")
    print()

n = len(CONVERSATIONAL_SAMPLE)
print(f"Raw follow-up hit rate:       {raw_hits}/{n}")
print(f"Rewritten (history-aware):    {rewritten_hits}/{n}")


OK  raw       | "How does that compare to MasterCard?"
OK  rewritten | (resolved via history_aware_retriever)

ERR raw       | "What was the actual 2017 figure?"
ERR rewritten | (resolved via history_aware_retriever)

ERR raw       | "What was the value in the earlier year?"
OK  rewritten | (resolved via history_aware_retriever)

ERR raw       | "Did it go up or down?"
OK  rewritten | (resolved via history_aware_retriever)

ERR raw       | "What about just 2012?"
OK  rewritten | (resolved via history_aware_retriever)

Raw follow-up hit rate:       1/5
Rewritten (history-aware):    4/5


## 8. Full comparison + save results

Same structure as your `eval_results_*.json` convention (`save_json` from
`src/utils/io.py`), so this drops straight into your existing
`data/processed/` folder alongside the Project 1-3 results.


In [ ]:
all_results = {
    "baseline": baseline_summary,
    "hyde": hyde_summary,
    "multi_query": mq_summary,
    "step_back": step_back_summary,
    "decompose": decompose_summary,
    "query_rewrite_conversational": {
        "n": len(CONVERSATIONAL_SAMPLE),
        "raw_followup_hit_rate": raw_hits / len(CONVERSATIONAL_SAMPLE),
        "rewritten_hit_rate": rewritten_hits / len(CONVERSATIONAL_SAMPLE),
    },
    "sample_size": len(sample),
    "sample_failure_mode_composition": dict(Counter(ex["failure_mode"] for ex in sample)),
}

save_json(all_results, "data/processed/eval_results_query_transform.json")
print(json.dumps(all_results, indent=2))


{
  "baseline": {
    "multihop": {
      "n": 24,
      "recall": 0.375,
      "precision": 0.19166666666666668,
      "avg_latency_s": 0.02278536558151245
    },
    "control": {
      "n": 34,
      "recall": 0.6470588235294118,
      "precision": 0.12941176470588237,
      "avg_latency_s": 0.02146007032955394
    },
    "vocab": {
      "n": 6,
      "recall": 0.16666666666666666,
      "precision": 0.03333333333333333,
      "avg_latency_s": 0.022030353546142578
    },
    "ALL": {
      "n": 64,
      "recall": 0.5,
      "precision": 0.14375000000000002,
      "avg_latency_s": 0.022010520100593567
    }
  },
  "hyde": {
    "multihop": {
      "n": 24,
      "recall": 0.125,
      "precision": 0.075,
      "avg_latency_s": 21.025333672761917
    },
    "control": {
      "n": 34,
      "recall": 0.4411764705882353,
      "precision": 0.08823529411764706,
      "avg_latency_s": 20.644519763834335
    },
    "vocab": {
      "n": 6,
      "recall": 0.0,
      "precision": 0.0,
   